In [1]:
import pandas as pd

# File paths
file1 = r"C:\Users\usuario\Downloads\Location1_train.csv"
file2 = r"C:\Users\usuario\Downloads\Location2_train.csv"
file3 = r"C:\Users\usuario\Downloads\Location3_train.csv"
file4 = r"C:\Users\usuario\Downloads\Location4_train.csv"  # change if needed

# Reading the CSV files
df1 = pd.read_csv(file1)
df2 = pd.read_csv(file2)
df3 = pd.read_csv(file3)
df4 = pd.read_csv(file4)

# Optional: display the first rows to verify
print(df1.head())
print(df2.head())
print(df3.head())
print(df4.head())

                  Time  temperature_2m  relativehumidity_2m  dewpoint_2m  \
0  2017-01-02 00:00:00            28.5                   85         24.5   
1  2017-01-02 01:00:00            28.4                   86         24.7   
2  2017-01-02 02:00:00            26.8                   91         24.5   
3  2017-01-02 03:00:00            27.4                   88         24.3   
4  2017-01-02 04:00:00            27.3                   88         24.1   

   windspeed_10m  windspeed_100m  winddirection_10m  winddirection_100m  \
0           1.44            1.26                146                 162   
1           2.06            3.99                151                 158   
2           1.30            2.78                148                 150   
3           1.30            2.69                 58                 105   
4           2.47            4.43                 58                  84   

   windgusts_10m   Power  
0            1.4  0.1635  
1            4.4  0.1424  
2          

In [ ]:
import pandas as pd
from typing import Tuple, Optional

def time_series_split(
    df: pd.DataFrame,
    time_col: str = "Time",
    train_ratio: float = 0.70,
    val_ratio: float = 0.15,
    test_ratio: float = 0.15,
) -> Tuple[pd.DataFrame, Optional[pd.DataFrame], pd.DataFrame]:
    """
    BEST-PRACTICE time-based split for chronological data (e.g. weather + Power forecasting).
    
    Why this is the best way:
    - Time series data MUST respect temporal order → never random shuffle (no future leakage).
    - Train = past, Validation = recent past, Test = future (real-world forecasting simulation).
    - Automatically sorts by Time and converts to datetime.
    - Flexible ratios (they must sum to 1.0).
    - Optional validation set (set val_ratio=0 to skip it).
    - Returns full DataFrames (including all columns: Time, temperature_2m, ..., Power) so you can
      easily create X_train / y_train = train.drop(columns=['Power']), etc.
    
    Example usage:
        train, val, test = time_series_split(your_df)
        # or without validation:
        train, test = time_series_split(your_df, val_ratio=0.0)
        
        # Then for modeling:
        X_train = train.drop(columns=['Power'])
        y_train = train['Power']
        # same for val / test
    """
    if not isinstance(df, pd.DataFrame):
        raise TypeError("Input must be a pandas DataFrame")
    if time_col not in df.columns:
        raise ValueError(f"Column '{time_col}' not found. Available columns: {list(df.columns)}")

    # Work on a copy so we don't modify the original
    data = df.copy()

    # Convert Time to datetime (handles strings, Unix timestamps, etc.)
    if not pd.api.types.is_datetime64_any_dtype(data[time_col]):
        data[time_col] = pd.to_datetime(data[time_col], errors="coerce")

    # Sort chronologically (critical!)
    data = data.sort_values(by=time_col).reset_index(drop=True)

    # Validate ratios
    if not abs(train_ratio + val_ratio + test_ratio - 1.0) < 1e-6:
        raise ValueError(f"Ratios must sum to 1.0 (got {train_ratio + val_ratio + test_ratio})")

    n = len(data)
    if n < 10:
        raise ValueError("Dataset too small for meaningful split")

    # Calculate exact indices (integer splits)
    train_end = int(n * train_ratio)
    val_end = int(n * (train_ratio + val_ratio))

    # Prevent empty sets
    if train_end == 0:
        train_end = max(1, int(n * 0.1))
    if val_end <= train_end and val_ratio > 0:
        val_end = train_end + max(1, int(n * 0.1))

    train_df = data.iloc[:train_end].copy()
    val_df = data.iloc[train_end:val_end].copy() if val_ratio > 0 else None
    test_df = data.iloc[val_end:].copy()

    print(f"✅ Time-series split completed:")
    print(f"   • Train:     {len(train_df):,} rows ({train_ratio:.0%}) → {train_df[time_col].min()} to {train_df[time_col].max()}")
    if val_df is not None:
        print(f"   • Validation: {len(val_df):,} rows ({val_ratio:.0%}) → {val_df[time_col].min()} to {val_df[time_col].max()}")
    print(f"   • Test:      {len(test_df):,} rows ({test_ratio:.0%})  → {test_df[time_col].min()} to {test_df[time_col].max()}")

    if val_df is not None:
        return train_df, val_df, test_df
    else:
        return train_df, test_df

In [4]:
df_i = [df1, df2, df3, df4]
for i in df_i:
    train, val, test = time_series_split(i)
    

✅ Time-series split completed:
   • Train:     24,528 rows (70%) → 2017-01-02 00:00:00 to 2019-10-20 23:00:00
   • Validation: 5,256 rows (15%) → 2019-10-21 00:00:00 to 2020-05-26 23:00:00
   • Test:      5,256 rows (15%)  → 2020-05-27 00:00:00 to 2020-12-31 23:00:00
✅ Time-series split completed:
   • Train:     24,528 rows (70%) → 2017-01-02 00:00:00 to 2019-10-20 23:00:00
   • Validation: 5,256 rows (15%) → 2019-10-21 00:00:00 to 2020-05-26 23:00:00
   • Test:      5,256 rows (15%)  → 2020-05-27 00:00:00 to 2020-12-31 23:00:00
✅ Time-series split completed:
   • Train:     24,528 rows (70%) → 2017-01-02 00:00:00 to 2019-10-20 23:00:00
   • Validation: 5,256 rows (15%) → 2019-10-21 00:00:00 to 2020-05-26 23:00:00
   • Test:      5,256 rows (15%)  → 2020-05-27 00:00:00 to 2020-12-31 23:00:00
✅ Time-series split completed:
   • Train:     24,528 rows (70%) → 2017-01-02 00:00:00 to 2019-10-20 23:00:00
   • Validation: 5,256 rows (15%) → 2019-10-21 00:00:00 to 2020-05-26 23:00:00
   • Te